In [1]:
import anndata as ad
import os
import requests
import scprep
import tempfile
import time
from utils import *

COLLECTION_ID = "0b9d8a04-bb9d-44da-aa27-705bb65b54eb"
DOMAIN = "cellxgene.cziscience.com"
API_BASE = f"https://api.{DOMAIN}"
METHOD_ALIASES = {"10x 3' v2": "droplet", "Smart-seq2": "facs"}

In [2]:
import anndata
import numpy as np
from util import *

def split_data(
    adata: anndata.AnnData, train_frac: float = 0.9, seed: int = 0
) -> anndata.AnnData:
    """Split data using molecular cross-validation.

    Stores "train" and "test" dataset using the AnnData.obsm property.
    """
    import scipy.sparse

    random_state = np.random.RandomState(seed)

    X = adata.X

    if scipy.sparse.issparse(X):
        X = np.array(X.todense())
    if np.allclose(X, X.astype(int)):
        X = X.astype(int)
    else:
        raise TypeError("Molecular cross-validation requires integer count data.")

    X_train, X_test = split_molecules(
        X, 0.9, 0.0, random_state
    )
    # remove zero entries
    is_missing = X_train.sum(axis=0) == 0
    X_train, X_test = X_train[:, ~is_missing], X_test[:, ~is_missing]

    adata = adata[:, ~is_missing].copy()
    adata.obsm["train"] = scipy.sparse.csr_matrix(X_train).astype(float)
    adata.obsm["test"] = scipy.sparse.csr_matrix(X_test).astype(float)

    return adata

# Lung

In [14]:
import scanpy as sc

lung = sc.read_h5ad('../../../oscb/uploads/tabula-muris-senis-droplet-processed-official-annotations-Lung.h5ad')
lung

/opt/conda/lib/python3.11/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/opt/conda/lib/python3.11/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


AnnData object with n_obs × n_vars = 24540 × 20138
    obs: 'age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden'
    var: 'n_cells', 'means', 'dispersions', 'dispersions_norm', 'highly_variable'
    uns: 'age_colors', 'cell_ontology_class_colors', 'leiden', 'louvain', 'neighbors', 'pca'
    obsm: 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [15]:
f'{None}'

'None'

In [10]:
def _get_json(url, retries=5, sleep=0.05, backoff=2):
    try:
        res = requests.get(url=url, headers={"Content-Type": "application/json"})
        return res.json()
    except Exception:  # pragma: nocover
        if retries > 0:
            time.sleep(sleep)
            return _get_json(url, retries - 1, sleep * backoff, backoff)
        raise


def check_unknown_organs(datasets, organ_list):
    known_organs = set([t["label"] for d in datasets for t in d["tissue"]])
    unknown_organs = set(organ_list) - known_organs
    if unknown_organs:
        raise ValueError(
            f"Unknown organs provided in `organ_list': {', '.join(unknown_organs)}."
            f" Known organs are {', '.join(known_organs)}"
        )


def matching_dataset(dataset, method_list, organ_list):
    # if dataset has multiple methods, skip it
    if len(dataset["assay"]) > 1:
        return False

    # if dataset has multiple tissues, skip it
    if len(dataset["tissue"]) > 1:
        return False

    method = dataset["assay"][0]["label"]
    method = METHOD_ALIASES[method]

    # if organ_list is not empty, check for specific tissue
    if len(organ_list) > 0 and dataset["tissue"][0]["label"] not in organ_list:
        return False

    # if method_list is not empty, check for specific method
    if len(method_list) > 0 and method not in method_list:
        return False

    return True


def load_raw_counts(dataset):
    import scanpy as sc
    
    print(dataset)
    dataset_id = dataset["id"]
    assets_path = (
        f"/curation/v1/collections/{COLLECTION_ID}/datasets/{dataset_id}/assets"
    )
    url = f"{API_BASE}{assets_path}"
    assets = _get_json(url)
    assets = [asset for asset in assets if asset["filetype"] == "H5AD"]
    assert len(assets) == 1
    asset = assets[0]

    filename = f"{COLLECTION_ID}_{dataset_id}_{asset['filename']}"
    with tempfile.TemporaryDirectory() as tempdir:
        filepath = os.path.join(tempdir, filename)
        scprep.io.download.download_url(asset["presigned_url"], filepath)
        adata = sc.read_h5ad(filepath)

    filter_genes_cells(adata)
    # If `raw` exists, raw counts are there
    if getattr(adata, "raw", None) is not None:
        return adata.raw.to_adata()
    return adata


def load_tabula_muris_senis(test=False, method_list=None, organ_list=None):
    """Load tubula_muris_senis datasets into 1 anndata object based on user input.

    Input which methods and organs to create anndata object from.
    Returns a single anndata object with specified methods and organs.
    EX: load_tabula_muris_senis(method_list = ['facs', 'droplet'],
    organ_list = ['Skin', 'Fat']), and returns anndata for facs-skin, droplet-skin,
    and droplet-fat anndata sets. (no facs-fat dataset available)
    """

    if method_list is None:
        method_list = []
    if organ_list is None:
        organ_list = []
    method_list = [x.lower() for x in method_list]
    organ_list = [x.lower() for x in organ_list]

    unknown_methods = set(method_list) - set(["facs", "droplet"])
    if unknown_methods:
        raise ValueError(
            f"Unknown methods provided in `method_list': {','.join(unknown_methods)}."
            " Known methods are `facs' and `droplet'"
        )

    datasets_path = f"/curation/v1/collections/{COLLECTION_ID}"
    url = f"{API_BASE}{datasets_path}"
    datasets = _get_json(url)["datasets"]
    check_unknown_organs(datasets, organ_list)

    adata_list = []
    for dataset in datasets:
        if matching_dataset(dataset, method_list, organ_list):
            adata_list.append(load_raw_counts(dataset))

    assert len(adata_list) > 0
    adata = ad.concat(adata_list, join="outer")

    # this obs key causes write errors
    del adata.obs["is_primary_data"]

    if test:
        adata = subsample_even(adata, n_obs=500, even_obs="method")
        adata = adata[:, :1000]
        filter_genes_cells(adata)
    return adata

In [11]:
lung = load_tabula_muris_senis(
        organ_list=["lung"],
        method_list=["droplet"],
        test=False,
    )
lung = split_data(lung)
lung

{'assay': [{'label': "10x 3' v2", 'ontology_term_id': 'EFO:0009899'}], 'assets': [{'filesize': 302325517, 'filetype': 'H5AD', 'url': 'https://datasets.cellxgene.cziscience.com/e818d27c-62ae-4c19-97c7-6cd4d65b8f9b.h5ad'}], 'cell_count': 24540, 'cell_type': [{'label': 'B cell', 'ontology_term_id': 'CL:0000236'}, {'label': 'CD4-positive, alpha-beta T cell', 'ontology_term_id': 'CL:0000624'}, {'label': 'CD8-positive, alpha-beta T cell', 'ontology_term_id': 'CL:0000625'}, {'label': 'T cell', 'ontology_term_id': 'CL:0000084'}, {'label': 'adventitial cell', 'ontology_term_id': 'CL:0002503'}, {'label': 'alveolar macrophage', 'ontology_term_id': 'CL:0000583'}, {'label': 'basophil', 'ontology_term_id': 'CL:0000767'}, {'label': 'bronchial smooth muscle cell', 'ontology_term_id': 'CL:0002598'}, {'label': 'classical monocyte', 'ontology_term_id': 'CL:0000860'}, {'label': 'club cell', 'ontology_term_id': 'CL:0000158'}, {'label': 'dendritic cell', 'ontology_term_id': 'CL:0000451'}, {'label': 'endothe

KeyError: 'id'

# 1k Peripheral blood mononuclear cells

In [3]:
PBMC_1K_URL = "https://ndownloader.figshare.com/files/36088667"
def load_tenx_1k_pbmc(test=False):
    """Download PBMC data from Figshare."""
    import scanpy as sc

    if test:
        adata = load_tenx_1k_pbmc(test=False)
        sc.pp.subsample(adata, n_obs=100)
        adata = adata[:, :1000]
        filter_genes_cells(adata)
    else:
        with tempfile.TemporaryDirectory() as tempdir:
            filepath = os.path.join(tempdir, "pbmc.h5ad")
            scprep.io.download.download_url(PBMC_1K_URL, filepath)
            adata = sc.read_h5ad(filepath)
            filter_genes_cells(adata)
    return adata

In [4]:
pbmc = load_tenx_1k_pbmc(test=False)
pbmc

AnnData object with n_obs × n_vars = 1087 × 15099
    obs: 'n_counts'
    var: 'n_cells'
    uns: 'var_names_all'

In [9]:
pbmc.write_h5ad('../../../oscb/pbmc_1k.h5ad')

In [17]:
pbmc = split_data(pbmc)
pbmc

AnnData object with n_obs × n_vars = 1087 × 15099
    obs: 'n_counts'
    var: 'n_cells'
    uns: 'var_names_all'
    obsm: 'train', 'test'

In [20]:
pbmc.obsm['train'].toarray()

array([[0., 0., 0., ..., 3., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 6., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [21]:
# Save the dense array to a CSV file
np.savetxt('../../../oscb/pbmc_1k.csv', pbmc.obsm['train'].toarray(), delimiter=",")

In [16]:
pbmc.X.todense()

matrix([[0., 0., 0., ..., 3., 1., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 8., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [15]:
pbmc.obsm['train'].todense()

matrix([[0., 0., 0., ..., 3., 1., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 6., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
pbmc.obsm['test'].todense().shape

(1087, 15099)

In [17]:
pbmc.obsm['test'].todense()

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 2., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

# Pancreas (inDrop)

In [6]:
URL = "https://ndownloader.figshare.com/files/36086813"

def load_pancreas(test=False, keep_techs=None):
    """Download pancreas data from figshare."""
    import scanpy as sc

    if test:
        # load full data first, cached if available
        adata = load_pancreas(
            test=False,
            keep_techs=keep_techs or ["celseq", "inDrop4", "smarter"],
        )

        keep_celltypes = adata.obs["celltype"].dtype.categories[[0, 3]]
        adata = adata[adata.obs["celltype"].isin(keep_celltypes)].copy()

        # Subsample pancreas data
        adata = adata[:, :500].copy()
        # Note: could also use 200-500 HVGs rather than 200 random genes
        filter_genes_cells(adata)

        # select 250 cells from each celltype
        keep_cell_idx = np.concatenate(
            [
                np.random.choice(
                    np.argwhere(adata.obs["celltype"].to_numpy() == ct).flatten(),
                    min(250, np.sum(adata.obs["celltype"].to_numpy() == ct)),
                    replace=False,
                )
                for ct in keep_celltypes
            ]
        )
        adata = adata[adata.obs.index[keep_cell_idx]]

        # Ensure there are no cells or genes with 0 counts
        filter_genes_cells(adata)

        return adata

    with tempfile.TemporaryDirectory() as tempdir:
        filepath = os.path.join(tempdir, "pancreas.h5ad")
        scprep.io.download.download_url(URL, filepath)
        adata = sc.read(filepath)

    if keep_techs is not None:
        adata = adata[adata.obs["tech"].isin(keep_techs)].copy()

    # NOTE: adata.X contains log-normalized data, so we're moving it
    adata.layers["log_normalized"] = adata.X
    adata.X = adata.layers["counts"]

    # Ensure there are no cells or genes with 0 counts
    filter_genes_cells(adata)

    return adata

In [7]:
pancreas = load_pancreas(test=False, keep_techs=["inDrop1"])
pancreas

AnnData object with n_obs × n_vars = 1937 × 15575
    obs: 'tech', 'celltype', 'size_factors', 'n_counts'
    var: 'n_cells'
    uns: 'var_names_all'
    layers: 'counts', 'log_normalized'

In [8]:
pancreas.write_h5ad('../../../oscb/pancreas.h5ad')

In [20]:
pancreas = split_data(pancreas)
pancreas

AnnData object with n_obs × n_vars = 1937 × 15513
    obs: 'tech', 'celltype', 'size_factors', 'n_counts'
    var: 'n_cells'
    uns: 'var_names_all'
    obsm: 'train', 'test'
    layers: 'counts', 'log_normalized'